# Four losing trades: transaction-level analysis

## tl;dr

Reconstructs entry, hold, exit, dump concentration, and recovery from the uploaded archive and confirmed fills.

## Context & Methods

Decision: determine whether losses point to entry quality, execution, or sell logic. Archive spot prices are SOL/token; bot prices are divided by 1,000 for comparison. Exact PNL comes from confirmed SOL flows. Same-second timestamps have one-second resolution; post-exit recovery is not an executable-fill backtest.

## Data and Results

In [1]:
from pathlib import Path
import json,zipfile
from collections import defaultdict
ROOT=Path("/home/vibes/nuxflix/8888")
MINTS=["EXMRz2urNJ5EwTVoeeHDBvJqiWmw3DSf1e9ZugQrpump","A6J99dEAf6wovckChrCEf53UD1jXRbUV7cRRccLCpump","2CkBXeqRM6YpjpuStn6NFhjsjVneRqiUwCCFnouhpump","CyR74ybdwRnNyft65ozaVSt933H2Yfd3XsoHwfWjpump"];TARGET="88887QrRZPZmsstEXsoXXB8E7nbmUDde4Gp5s7jG3ENu"
P={r["mint"]:r for r in map(json.loads,(ROOT/"logs/pnl.jsonl").read_text().splitlines()) if r.get("mint") in MINTS};L=defaultdict(list)
for r in map(json.loads,(ROOT/"recovery/lifecycle.jsonl").read_text().splitlines()):
 m=r.get("descriptor",{}).get("mint")
 if m in MINTS:L[m].append(r)
A={}
with zipfile.ZipFile(ROOT/"EqdQgjXzYu1GZ4XKkTqmf8CgZkCbhAkX9bkJv9CrDJ3K.zip") as z:
 for m in MINTS:A[m]=json.loads(z.read(next(x for x in z.namelist() if x.endswith("/"+m+".json"))))
def ix(es,s):return next((i for i,e in enumerate(es) if e["signature"]==s),None)
def pct(a,b):return (a/b-1)*100 if b else None
def flow(es):
 b=sum(e["sol_amount"] for e in es if e["side"]=="BUY");s=sum(e["sol_amount"] for e in es if e["side"]=="SELL");return b,s,b/(b+s) if b+s else None
def sellers(es):
 d=defaultdict(set)
 for e in es:
  if e["side"]=="SELL":d[e["slot"]].add(e["wallet"])
 return max(map(len,d.values()),default=0)
rows=[];D={}
for m in MINTS:
 r=P[m];es=A[m]["events"];bi=ix(es,r["buySignature"]);si=ix(es,r["sellSignature"]);assert bi is not None and si is not None
 be,se=es[bi],es[si];tsi=max(i for i,e in enumerate(es[:bi]) if e["wallet"]==TARGET and e["side"]=="SELL");ts=es[tsi]
 held=es[bi:si+1];mx=max(held,key=lambda e:e["price"]);mn=min(held,key=lambda e:e["price"]);hb,hs,hp=flow(held)
 pre=[e for e in es[:si] if se["timestamp"]-2<=e["timestamp"]<=se["timestamp"]];pb,ps,pp=flow(pre)
 p60=[e for e in es[si+1:] if e["timestamp"]<=se["timestamp"]+60];p300=[e for e in es[si+1:] if e["timestamp"]<=se["timestamp"]+300]
 conf=[e for e in es[tsi+1:] if e["timestamp"]<=ts["timestamp"]+2];cb,cs,cp=flow(conf);cpr=[e["price"] for e in conf] or [ts["price"]]
 rows.append(dict(token=m[:6],hold_s=r["holdingTimeMs"]/1000,entry_mc=r["entryMarketCapSol"],entry_dev=r["prices"]["actualEntryDeviationPct"],signal_pnl=pct(r["prices"]["exitSignalPrice"],r["entryPrice"]),realized_pnl=r["pnlPct"],pnl_sol=r["pnlSol"],mfe=pct(mx["price"],r["entryPrice"]/1000),mae=pct(mn["price"],r["entryPrice"]/1000),held_buy_pressure=hp,pre2_sell_sol=ps,pre2_sell_pressure=None if pp is None else 1-pp,same_slot_sellers=sellers(pre),recovery60=None if not p60 else pct(max(e["price"] for e in p60),r["exitPrice"]/1000),recovery300=None if not p300 else pct(max(e["price"] for e in p300),r["exitPrice"]/1000)))
 D[m]=dict(target_sell_sol=ts["sol_amount"],target_sell_curve=ts["bonding_curve_progress"],confirm_n=len(conf),confirm_pressure=cp,confirm_drawdown=pct(min(cpr),ts["price"]),confirm_return=pct(cpr[-1],ts["price"]),buy_attempts=sum(x["event"]=="buy_sent" for x in L[m]),sell_attempts=sum(x["event"]=="sell_sent" for x in L[m]),mfe_after_s=mx["timestamp"]-be["timestamp"],mae_after_s=mn["timestamp"]-be["timestamp"],full_recovery=None if not es[si+1:] else pct(max(e["price"] for e in es[si+1:]),r["exitPrice"]/1000),archive_events=len(es))
print(json.dumps(rows,indent=2))


[
  {
    "token": "EXMRz2",
    "hold_s": 29.285,
    "entry_mc": 123.9307427173895,
    "entry_dev": 8.415198330480166,
    "signal_pnl": -41.83370828457543,
    "realized_pnl": -39.33553412623366,
    "pnl_sol": -0.0038554,
    "mfe": 1.6010565158648626,
    "mae": -41.237413884959594,
    "held_buy_pressure": 0.41201278406426134,
    "pre2_sell_sol": 4.867984529,
    "pre2_sell_pressure": 0.684078131696439,
    "same_slot_sellers": 5,
    "recovery60": 10.59649456386531,
    "recovery300": 10.59649456386531
  },
  {
    "token": "A6J99d",
    "hold_s": 36.968,
    "entry_mc": 81.80895318029133,
    "entry_dev": 7.564599162598529,
    "signal_pnl": -41.09906514904325,
    "realized_pnl": -43.46503531922193,
    "pnl_sol": -0.004269377,
    "mfe": 2.60317656415614,
    "mae": -43.469089293343586,
    "held_buy_pressure": 0.37411354006822883,
    "pre2_sell_sol": 4.796395071,
    "pre2_sell_pressure": 0.9982673053411678,
    "same_slot_sellers": 4,
    "recovery60": 107.94498547906927

In [2]:
for m in MINTS: print("\n"+m+"\n"+json.dumps(D[m],indent=2))
checks=[dict(token=m[:6],recomputed=int(P[m]["sellLamports"])-int(P[m]["buyLamports"]),recorded=int(P[m]["pnlLamports"]),pct_recomputed=(int(P[m]["sellLamports"])/int(P[m]["buyLamports"])-1)*100,pct_recorded=P[m]["pnlPct"],signatures_present=ix(A[m]["events"],P[m]["buySignature"]) is not None and ix(A[m]["events"],P[m]["sellSignature"]) is not None) for m in MINTS]
print("\nCHECKS\n"+json.dumps(checks,indent=2))


EXMRz2urNJ5EwTVoeeHDBvJqiWmw3DSf1e9ZugQrpump
{
  "target_sell_sol": 2.264365203,
  "target_sell_curve": 0.5760545429188249,
  "confirm_n": 13,
  "confirm_pressure": 0.9557431872983769,
  "confirm_drawdown": -3.9728668396756928,
  "confirm_return": 33.3729381155611,
  "buy_attempts": 2,
  "sell_attempts": 1,
  "mfe_after_s": 1,
  "mae_after_s": 29,
  "full_recovery": 10.59649456386531,
  "archive_events": 568
}

A6J99dEAf6wovckChrCEf53UD1jXRbUV7cRRccLCpump
{
  "target_sell_sol": 1.313684819,
  "target_sell_curve": 0.5274265229797327,
  "confirm_n": 7,
  "confirm_pressure": 0.9006703185765284,
  "confirm_drawdown": -2.404899355812762,
  "confirm_return": -0.7779383164541298,
  "buy_attempts": 2,
  "sell_attempts": 2,
  "mfe_after_s": 7,
  "mae_after_s": 36,
  "full_recovery": 140.29135970103584,
  "archive_events": 492
}

2CkBXeqRM6YpjpuStn6NFhjsjVneRqiUwCCFnouhpump
{
  "target_sell_sol": 1.217175325,
  "target_sell_curve": 0.5347956991338734,
  "confirm_n": 15,
  "confirm_pressure": 0.

## Takeaways

Interpret entry execution, signal-versus-fill loss, sell concentration, and recovery together. See the chat summary for decision implications.